In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys

import matplotlib.pyplot as plt

sys.path.append("/home/johmathe/bgbench")
os.environ["PYTHONPATH"] = os.pathsep.join(sys.path)
from src.data import hf_datamodule

In [ ]:
# TODO: Rework.
datamodule = hf_datamodule.MotrPacDataModule(
    data_dir="data/proteomics/",
    batch_size=654,
    num_workers=0,
    pin_memory=False,
    n_selected_nodes=100,
    imputation_method="mean",
)
if not os.path.exists(datamodule.hparams.data_dir):
    os.makedirs(datamodule.hparams.data_dir)
print("preparing data")
datamodule.prepare_data()
print("setting up data")
datamodule.setup()
print("creating dataloader")
dataloader = datamodule.train_dataloader()
print("preparing dataset")
raw_data, targets = datamodule.prepare_dataset()

In [ ]:
import torch
import transforms

idx = datamodule.select_nodes(raw_data.values, targets, 100)
X_np = raw_data.iloc[:, idx].values
y_np = targets
X_np = X_np[:457]
y_np = y_np[:457]
feature_normalizer = transforms.MeanStdNormalizer()
target_normalizer = transforms.MeanStdNormalizer()

feature_normalizer.fit(X_np)
target_normalizer.fit(y_np.reshape(-1, 1))
X = torch.from_numpy(feature_normalizer.transform(X_np)).to(torch.float32)
y = torch.from_numpy(target_normalizer.transform(y_np.reshape(-1, 1))).to(torch.float32)

In [ ]:
# Initialize model and training parameters
from src.models.naive_models import MLP4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLP4(in_channels=100, out_channels=1, hidden_channels=1024, num_layers=4).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.MSELoss()

# Training loop
num_epochs = 20
losses = []

for epoch in range(num_epochs):
    for batch in dataloader:
        batch = batch.to(device)
        optimizer.zero_grad()
        predictions = model(batch.x, None, None)
        loss = criterion(predictions, batch.y)
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}")
        losses.append(loss.item())

In [ ]:
num_epochs = 1
for epoch in range(num_epochs):
    for a in dataloader:
        print((a.x.to(device) - X.flatten().to(device)).max())
        print(a.y.to(device) - y.flatten().to(device))

In [ ]:
# Training loop
num_epochs = 1000
losses = []

# Move data to GPU
X = X.to(device)
y = y.to(device)

# Reset optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(num_epochs):
    optimizer.zero_grad()
    predictions = model(
        X.flatten(), None, None
    )  # No adjacency matrix or batch vector needed for MLP
    loss = criterion(predictions, y)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")
    losses.append(loss.item())

# Plot training loss
plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()